# Using Lacuna with Docker

This guide demonstrates how to run Lacuna analyses using the pre-built Docker image, without installing Lacuna or its dependencies locally. We run a structural network mapping (SNM) analysis using the HCP1065 tractogram as a complete example.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/how-to/docker.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Prerequisites

You need Docker installed on your system. See [Get Docker](https://docs.docker.com/get-docker/) for installation instructions.

The Lacuna Docker image includes all dependencies (Python, MRtrix3, TemplateFlow templates) so you do not need to install anything else.

## Setup

Pull the Lacuna Docker image.

In [1]:
!docker pull ghcr.io/lacuna/lacuna:latest

Error response from daemon: Head "https://ghcr.io/v2/lacuna/lacuna/manifests/latest": denied


Verify the image works.

In [ ]:
!docker run --rm ghcr.io/lacuna/lacuna:latest --help

## How volume mounts work

Docker containers are isolated from the host filesystem. To give the container access to your data, you mount host directories into the container using `-v`:

```
-v /host/path:/container/path:ro   # read-only
-v /host/path:/container/path      # read-write
```

| Purpose | Host path | Container path | Mode |
|---------|-----------|----------------|------|
| BIDS input data | `/path/to/bids` | `/bids` | read-only (`:ro`) |
| Output results | `/path/to/output` | `/output` | read-write |
| Connectomes | `/path/to/connectomes` | `/connectomes` | read-only (`:ro`) |

The container entrypoint is `lacuna`, so you pass subcommands directly after the image name.

## Prepare tutorial data

We use Lacuna inside the container to create a tutorial dataset. The `--rm` flag removes the container after it exits.

In [ ]:
!docker run --rm \
    -v /tmp/docker_tutorial:/data \
    ghcr.io/lacuna/lacuna:latest \
    tutorial /data/bids --force

In [ ]:
!ls /tmp/docker_tutorial/bids/

## Fetch the HCP1065 tractogram

Download the HCP1065 structural connectome. This is a lightweight tractogram (~300 MB) suitable for structural network mapping.

In [ ]:
!docker run --rm \
    -v /tmp/docker_tutorial:/data \
    ghcr.io/lacuna/lacuna:latest \
    fetch hcp1065 --output-dir /data/connectomes

In [ ]:
!ls /tmp/docker_tutorial/connectomes/

## Run structural network mapping

Run the SNM analysis on a single subject. Note how the BIDS input is mounted read-only (`:ro`) while the output directory is read-write.

In [ ]:
!docker run --rm \
    -v /tmp/docker_tutorial/bids:/bids:ro \
    -v /tmp/docker_tutorial/connectomes:/connectomes:ro \
    -v /tmp/docker_tutorial/output:/output \
    ghcr.io/lacuna/lacuna:latest \
    run snm /bids /output \
    --connectome-path /connectomes/hcp1065.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --verbose

Inspect the output. The results are written to the host filesystem via the volume mount.

In [ ]:
!ls /tmp/docker_tutorial/output/sub-01/ses-01/anat/

## Visualize results

The output files are standard NIfTI images on the host filesystem, so we can visualize them with nilearn without Docker.

In [ ]:
!pip install nibabel nilearn

In [ ]:
import nibabel as nib
from nilearn import plotting

# Load the disconnectivity map
discon_path = "/tmp/docker_tutorial/output/sub-01/ses-01/anat/sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_desc-snm_disconnectivitymap.nii.gz"
discon_img = nib.load(discon_path)

plotting.plot_stat_map(discon_img,
                       radiological=True,
                       title="Structural disconnectivity map (Docker)",
                       draw_cross=False,
                       colorbar=True)

## Run on all subjects

Omit `--participant-label` to process every subject in the BIDS dataset.

In [ ]:
!docker run --rm \
    -v /tmp/docker_tutorial/bids:/bids:ro \
    -v /tmp/docker_tutorial/connectomes:/connectomes:ro \
    -v /tmp/docker_tutorial/output:/output \
    ghcr.io/lacuna/lacuna:latest \
    run snm /bids /output \
    --connectome-path /connectomes/hcp1065.tck \
    --mask-space MNI152NLin6Asym

## Collect results

Aggregate individual outputs into group-level tables.

In [ ]:
!docker run --rm \
    -v /tmp/docker_tutorial/output:/output \
    ghcr.io/lacuna/lacuna:latest \
    collect /output \
    --output-dir /output/group

## Resource limits

For large datasets or tractograms, you can control the resources available to the container.

In [ ]:
# Limit to 4 CPUs and 16 GB RAM
!docker run --rm \
    --cpus="4" \
    --memory="16g" \
    -v /tmp/docker_tutorial/bids:/bids:ro \
    -v /tmp/docker_tutorial/connectomes:/connectomes:ro \
    -v /tmp/docker_tutorial/output_limited:/output \
    ghcr.io/lacuna/lacuna:latest \
    run snm /bids /output \
    --connectome-path /connectomes/hcp1065.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --nprocs 4

## Tips

- Always mount input data as read-only (`:ro`) to prevent accidental modification.
- Mount the connectome directory separately from the BIDS input for clarity.
- Use `--rm` to automatically clean up containers after they exit.
- For HPC environments, consider using [Apptainer](https://apptainer.org/) (formerly Singularity) instead — it can pull the same Docker image with `apptainer pull docker://ghcr.io/lacuna/lacuna:latest`.
- The container includes pre-fetched TemplateFlow templates, so no internet access is needed at runtime for spatial transformations.